In [11]:
'''
Financial Sentiment Classification using FinBERT
Step 6 - Error Analysis

Goal: Inspect the fine-tuned FinBERT model's mistakes on the test set to
understand WHY it fails, not just report that it got 88% macro F1.
Ties back to the negation/numbers checks done in Step 1 EDA.

Input: test_predictions.csv (from Step 5, Colab) - contains each test
sentence, its true label, FinBERT's predicted label, and a correctness flag.

Test set results (Step 5): accuracy 89.26%, macro F1 88.03%.
Largest confusion: positive misclassified as neutral (22 cases).
'''

"\nFinancial Sentiment Classification using FinBERT\nStep 6 - Error Analysis\n\nGoal: Inspect the fine-tuned FinBERT model's mistakes on the test set to\nunderstand WHY it fails, not just report that it got 88% macro F1.\nTies back to the negation/numbers checks done in Step 1 EDA.\n\nInput: test_predictions.csv (from Step 5, Colab) - contains each test\nsentence, its true label, FinBERT's predicted label, and a correctness flag.\n\nTest set results (Step 5): accuracy 89.26%, macro F1 88.03%.\nLargest confusion: positive misclassified as neutral (22 cases).\n"

In [12]:
from pathlib import Path
import pandas as pd

df = pd.read_csv("../test_predictions.csv")  # adjust path to wherever you saved it
print(df.shape)
print(df["is_correct"].value_counts())

(484, 6)
is_correct
True     432
False     52
Name: count, dtype: int64


In [13]:
LABELS = {0: "negative", 1: "neutral", 2: "positive"}

errors = df[~df["is_correct"]].copy()
errors["true_name"] = errors["label"].map(LABELS)
# predicted_label_name already exists from Step 5

error_breakdown = errors.groupby(["true_name", "predicted_label_name"]).size().sort_values(ascending=False)
print(error_breakdown)

true_name  predicted_label_name
positive   neutral                 22
neutral    positive                14
           negative                 8
negative   neutral                  6
           positive                 1
positive   negative                 1
dtype: int64


In [16]:
import re

NEGATIONS = r"\b(not|no|never|neither|without|fails?|failing|failed)\b"
df["has_negation"] = df["sentence"].str.contains(NEGATIONS, case=False, regex=True)
df["has_number"] = df["sentence"].str.contains(r"\d", regex=True)

print("Error rate overall:        ", (~df["is_correct"]).mean().round(3))
print("Error rate (has negation): ", (~df[df["has_negation"]]["is_correct"]).mean().round(3))
print("Error rate (no negation):  ", (~df[~df["has_negation"]]["is_correct"]).mean().round(3))
print()
print("Error rate (has number):   ", (~df[df["has_number"]]["is_correct"]).mean().round(3))
print("Error rate (no number):    ", (~df[~df["has_number"]]["is_correct"]).mean().round(3))

Error rate overall:         0.107
Error rate (has negation):  0.0
Error rate (no negation):   0.111

Error rate (has number):    0.094
Error rate (no number):     0.122


In [17]:
print(df["has_negation"].sum(), "out of", len(df), "test sentences contain negation")

15 out of 484 test sentences contain negation


In [15]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [21]:
# Recreate 'errors' now that has_negation/has_number exist on df
errors = df[~df["is_correct"]].copy()
errors["true_name"] = errors["label"].map(LABELS)

pd.set_option("display.max_colwidth", 150)
error_sample = errors[["sentence", "true_name", "predicted_label_name", "has_negation", "has_number"]]
print(error_sample.sample(10, random_state=42).to_string(index=False))

                                                                                                                                                                                                                         sentence true_name predicted_label_name  has_negation  has_number
The company reported a profit of 800,000 euro ($ 1.2 mln)on the sale of its Varesvuo Partners sub-group and a loss of 400,000 euro $ 623,000 caused by the sale of its program production subsidiary Oy Filmiteollisuus Fine Ab .   neutral             negative         False        True
                                                                                                                                              Joint procurement will be later extended to the factories in the Baltic countries .   neutral             positive         False       False
                                                                                                                                                   `` M

In [ ]:
'''
## Error Analysis — Findings

**Overall:** 52 errors out of 484 test sentences (10.7% error rate), consistent
with the reported test accuracy of 89.26%.

### Error distribution by direction

| True → Predicted | Count |
|---|---|
| Positive → Neutral | 22 |
| Neutral → Positive | 14 |
| Neutral → Negative | 8 |
| Negative → Neutral | 6 |
| Negative → Positive | 1 |
| Positive → Negative | 1 |

**Key observation:** 69% of all errors (36/52) involve the neutral class in
some direction. The model almost never flips sentiment entirely
(positive↔negative errors: only 2 total) — when it's wrong, it's wrong about
*degree* (missing or over-reading mild sentiment), not direction. This tracks
with the EDA finding that neutral sentences are often boilerplate/factual
language, making the neutral boundary inherently fuzzy — likely for human
annotators too, since labels are themselves majority votes.

### Cross-checks against EDA hypotheses

| Pattern | Error rate | Sample size | Verdict |
|---|---|---|---|
| Contains negation | 0.0% | 15 sentences | Promising, but sample too small to be conclusive |
| No negation | 11.1% | 469 sentences | — |
| Contains a number | 9.4% | — | Numbers *help*, not hurt — numeric figures are usually attached to clear directional language ("fell 12%") |
| No number | 12.2% | — | — |

This is notable: the project brief anticipated negation and numeric figures
as likely sources of error, but FinBERT handles both reasonably well — a
more interesting and defensible finding than confirming the expected
difficulty.

### Representative errors

| Sentence | Actual | Predicted | Possible reason |
|---|---|---|---|
| "...reported a profit of 800,000 euro... and a loss of 400,000 euro..." | neutral | negative | Mixed positive/negative sentiment in one sentence; model weighted the loss more heavily |
| "The executive group will participate in the adjustments with a fixed-term 5% salary cut." | positive | neutral | Investor-perspective mismatch — cost-cutting is favorable for shareholders, but "salary cut" reads negatively on the surface |
| "WIG index ended Thursday's session 0.1 pct up... WIG20 was 1.13 down..." | neutral | positive | Mixed signal (both "up" and "down"); general market report, not company-specific |
| "Joint procurement will be later extended to the factories in the Baltic countries." | neutral | positive | Administrative/procedural language misread as positive ("extended" sounds expansion-positive) |
| "The new agreement...involves the transfer of certain engineering and documentation functions..." | positive | neutral | Implicit positive sentiment (expanded cooperation) stated in dry administrative language, no explicit positive words |
| "YIT has chosen Telenor and Elisa as its principal suppliers..." | positive | neutral | Real business win stated factually, without sentiment-anchoring language |

### Summary

The dominant failure mode is **implicit, context-dependent sentiment** —
cases where the correct label depends on understanding business implications
(a cost cut is good for investors, being chosen as a supplier is a win) rather
than on explicit sentiment vocabulary. This is consistent with Section 8 of
the project plan, which flagged "context-dependent sentiment" as an expected
challenge. The negation and numeric-figure concerns, by contrast, turned out
to be largely handled well by FinBERT — likely because WordPiece tokenization
and self-attention let it model these patterns directly, unlike the baseline's
bag-of-words representation.
'''